In [9]:
!pip3 install statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 29.7 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 31.5 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [statsmodels] [statsmodels]


In [12]:
import nbformat
print(nbformat.__version__)

5.10.4


In [ ]:
import pandas as pd
import plotly.express as px

# Load dataset"/Users/hetvi/Downloads/enriched_listings.csv"
df = pd.read_csv("/Users/hetvi/Downloads/enriched_listings.csv")

# Clean data
df = df.dropna(subset=["lat", "lon", "price_per_bed"])
df = df[df["price_per_bed"] > 0]

# Create bubble map
fig = px.scatter_map(
    df,
    lat="lat",
    lon="lon",
    color="price_per_bed",       # affordability per student
    size="bedrooms",             # larger bubble = more roommates possible
    hover_name="complex_name",
    hover_data={
        "price_total": True,
        "price_per_bed": ':.0f',
        "bedrooms": True,
        "baths": True,
        "sqft": True,
        "neighborhood": True,
        "dist_to_memorial_union_mu": ':.2f',
        "lat": False,
        "lon": False
    },
    zoom=12,
    center={"lat": 38.5449, "lon": -121.7405},
    title="Aggie Housing Intelligence Engine: Cost per Student Map"
)

fig.update_layout(
    map_style="open-street-map",
    margin={"r":0, "t":50, "l":0, "b":0}
)

fig.show()

In [ ]:
#removed studio apartment listings and outliers for better visualization
import pandas as pd
import plotly.express as px

df = pd.read_csv("/Users/hetvi/Downloads/enriched_listings.csv")

df = df.dropna(subset=[
    "dist_to_memorial_union_mu",
    "price_per_bed",
    "bedrooms",
    "complex_name"
])

df = df[
    (df["price_per_bed"] > 0) &
    (df["dist_to_memorial_union_mu"] >= 0)
].copy()

# Bedroom grouping
def categorize_bedrooms(x):
    if x == 1:
        return "1 Bedroom"
    elif x == 2:
        return "2 Bedrooms"
    elif x == 3:
        return "3 Bedrooms"
    else:
        return "4+ Bedrooms"

df["bedroom_group"] = df["bedrooms"].apply(categorize_bedrooms)

fig = px.scatter(
    df,
    x="dist_to_memorial_union_mu",
    y="price_per_bed",
    color="bedroom_group",
    size="bedrooms",  # bubble size based on bedrooms
    size_max=22,
    opacity=0.7,
    hover_name="complex_name",
    hover_data={
        "price_total": ":.0f",
        "bedrooms": True,
        "baths": True,
        "sqft": ":.0f",
        "neighborhood": True,
        "dist_to_memorial_union_mu": ":.2f"
    },
    category_orders={
        "bedroom_group": [
            "1 Bedroom",
            "2 Bedrooms",
            "3 Bedrooms",
            "4+ Bedrooms"
        ]
    },
    title="Student Housing Cost vs Distance from UC Davis Campus",
    labels={
        "dist_to_memorial_union_mu": "Distance to Memorial Union (miles)",
        "price_per_bed": "Price per Bedroom ($)",
        "bedroom_group": "Unit Type"
    }
)

fig.update_layout(
    template="plotly_white",
    width=1100,
    height=650,
    title=dict(x=0.5),
    font=dict(size=16)
)

fig.show()

In [18]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# -----------------------------
# 1. Load housing dataset
# -----------------------------
df = pd.read_csv("/Users/hetvi/Downloads/enriched_listings.csv")

cols = [
    "lat", "lon", "complex_name", "bedrooms", "price_per_bed",
    "nearest_campus_dist", "nearest_grocery_dist",
    "dist_to_downtown_davis_3rd_and_g_st"
]

df = df.dropna(subset=cols).copy()
df = df[df["price_per_bed"] > 0]

# -----------------------------
# 2. Build student score
# -----------------------------
def normalize(series):
    return (series - series.min()) / (series.max() - series.min())

df["rent_score"] = (1 - normalize(df["price_per_bed"])) * 100
df["campus_score"] = (1 - normalize(df["nearest_campus_dist"])) * 100
df["amenity_score"] = (
    0.6 * (1 - normalize(df["nearest_grocery_dist"])) +
    0.4 * (1 - normalize(df["dist_to_downtown_davis_3rd_and_g_st"]))
) * 100

df["student_score"] = (
    0.5 * df["rent_score"] +
    0.3 * df["campus_score"] +
    0.2 * df["amenity_score"]
)

# -----------------------------
# 3. Your places of interest
# -----------------------------
PLACES_OF_INTEREST = [
    # --- campus buildings ---
    {
        "name": "Memorial Union (MU)",
        "lat": 38.5423,
        "lon": -121.7496,
        "category": "campus",
        "description": "main student union building, center of campus life"
    },
    {
        "name": "Silo",
        "lat": 38.5382,
        "lon": -121.7527,
        "category": "campus",
        "description": "food court area between classes, south side of campus"
    },
    {
        "name": "Shields Library",
        "lat": 38.5397,
        "lon": -121.7495,
        "category": "campus",
        "description": "main library - where everyone studies"
    },
    {
        "name": "ARC (Activities & Recreation Center)",
        "lat": 38.5428,
        "lon": -121.7591,
        "category": "campus",
        "description": "gym / rec center on west side of campus"
    },
    {
        "name": "Student Health Center",
        "lat": 38.5427,
        "lon": -121.7615,
        "category": "campus",
        "description": "health and counseling services"
    },

    # --- grocery stores ---
    {
        "name": "Trader Joes",
        "lat": 38.5467,
        "lon": -121.7615,
        "category": "grocery",
        "description": "885 Russell Blvd - closest TJs to campus"
    },
    {
        "name": "Safeway (North)",
        "lat": 38.5621,
        "lon": -121.7663,
        "category": "grocery",
        "description": "1451 W Covell Blvd - north davis safeway"
    },
    {
        "name": "Nugget Markets (East Covell)",
        "lat": 38.5610,
        "lon": -121.7331,
        "category": "grocery",
        "description": "1414 E Covell Blvd - nicer grocery store, bit pricey"
    },
    {
        "name": "Davis Food Co-op",
        "lat": 38.5493,
        "lon": -121.7399,
        "category": "grocery",
        "description": "620 G St - organic/local grocery downtown"
    },
    {
        "name": "Target",
        "lat": 38.5543,
        "lon": -121.6997,
        "category": "grocery",
        "description": "4601 2nd St - east davis, only target in town"
    },

    # --- social / food / fun ---
    {
        "name": "Downtown Davis (3rd & G St)",
        "lat": 38.5448,
        "lon": -121.7389,
        "category": "social",
        "description": "heart of downtown - restaurants, bars, woodstocks pizza"
    },
    {
        "name": "Davis Farmers Market",
        "lat": 38.5447,
        "lon": -121.7440,
        "category": "social",
        "description": "central park - sat mornings and wed afternoons"
    },

    # --- transit ---
    {
        "name": "Davis Amtrak Station",
        "lat": 38.5435,
        "lon": -121.7377,
        "category": "transit",
        "description": "train station downtown - capitol corridor to sac/bay"
    },
]

places_df = pd.DataFrame(PLACES_OF_INTEREST)

# -----------------------------
# 4. Base housing map
# -----------------------------
fig = px.scatter_map(
    df,
    lat="lat",
    lon="lon",
    color="student_score",
    size="bedrooms",
    size_max=22,
    zoom=12,
    center={"lat": 38.5449, "lon": -121.7405},
    hover_name="complex_name",
    hover_data={
        "price_per_bed": ":.0f",
        "rent_score": ":.1f",
        "campus_score": ":.1f",
        "amenity_score": ":.1f",
        "student_score": ":.1f",
        "bedrooms": True,
        "nearest_campus_dist": ":.2f",
        "nearest_grocery_dist": ":.2f",
        "lat": False,
        "lon": False
    },
    title="Aggie Housing Intelligence Engine: Student Value Score Map"
)

# -----------------------------
# 5. Add only your POI points
# -----------------------------
category_symbols = {
    "campus": "star",
    "grocery": "marker",
    "social": "circle",
    "transit": "square"
}

for category in places_df["category"].unique():
    subset = places_df[places_df["category"] == category]

    fig.add_trace(
        go.Scattermap(
            lat=subset["lat"],
            lon=subset["lon"],
            mode="markers+text",
            text=subset["name"],
            textposition="top right",
            marker=dict(
                size=14,
                symbol=category_symbols.get(category, "marker")
            ),
            name=category.capitalize(),
            customdata=subset[["description"]],
            hovertemplate="<b>%{text}</b><br>%{customdata[0]}<extra></extra>"
        )
    )

# -----------------------------
# 6. Style
# -----------------------------
fig.update_layout(
    map_style="open-street-map",
    template="plotly_white",
    margin={"r": 0, "t": 60, "l": 0, "b": 0},
    title=dict(x=0.5, xanchor="center"),
    legend_title_text="Map Layers"
)

fig.show()

In [20]:
!pip3 install streamlit

  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 23.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.4/795.4 kB 17.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 24.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 23.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.2/34.2 MB 24.1 MB/s  0:00:01 eta 0:00:01
Using cached tzdata-2025.3-py2.py3-none-any.whl (348 kB)
  Attempting uninstall: pandas━━━╺━━━━━━━━━━━━━━ 10/16 [pydeck]]
    Found existing installation: pandas 3.0.090m━━━━━━━━━━━━━━ 10/16 [pydeck]
    Uninstalling pandas-3.0.0:╺━━━━━━━━━━━━━━ 10/16 [pydeck]
      Successfully uninstalled pandas-3.0.0━━━━━━━━━━━━━━ 10/16 [pydeck]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [streamlit]16 [streamlit]
